In [15]:
# ================================================================
# MASTER CALENDAR AGING ANALYSIS CODE — VOLTAIQ ANALYTICS STUDIO
# ================================================================
# Analyses included:
#   1. Calendar Aging Series — SOH, capacity, energy, RTE,
#      coulombic efficiency, energy inefficiency, relative metrics,
#      capacity & time-weighted ΔV, max charge/discharge temperature,
#      ambient temperature
#   2. DCIR — optional; 10s DCR at 50% SOC, SOC-weighted 10s DCR,
#             relative versions; raw ΔV and resistance at 1s, 10s,
#             60s across all SOC levels
#
# Key differences from the Cycle Life master code:
#   • Data organised by MONTH (calendar time), not by cycle number.
#   • Positional alignment — index 0 in every month's list always
#     refers to the same physical cell. Use None for missing check-ups.
#   • All relative metrics normalise to ref_month (default "0M").
#   • Export: one sheet per cell (rows = months) + SOH Summary table.
#
# Instructions:
#   1. Edit CELL 2 (User Configuration) before running anything.
#   2. Run cells in order: Cell 1 → Cell 7.
# ================================================================
 
 
# ================================================================
# CELL 1 — IMPORTS
# ================================================================
import voltaiq_studio as vs
import re
import numpy as np
import pandas as pd
from scipy import interpolate

In [16]:
# ================================================================
# CELL 2 — USER CONFIGURATION
# Edit ALL values in this cell before running the analysis.
# ================================================================
 
# --- Test names per month ---
# Keys are month labels (e.g. "0M", "1M", "12M").
# Each value is a list with one entry per cell; position is FIXED
# across every month — index 0 always refers to the same cell.
# Use None where a check-up was not performed for a cell that month.
test_names_by_month = {
    "0M": [
        'CV_0015_CS2E0060_Pr_CaLT1000M_25C',
        'CV_0015_CS2E0023_Pr_CaLT1000M_25C',
        'CV_0015_CS2E0066_Pr_CaLT1000M_45C',
        'CV_0015_CS2E0067_Pr_CaLT1000M_45C',
        'CV_0015_CS2E0038_Pr_CaLT1000M_60C',
        'CV_0015_CS2E0042_Pr_CaLT1000M_60C'
    ],
    "1M": [
        'CV_0015_CS2E0060_Pr_CaLT1001M_25C',
        'CV_0015_CS2E0023_Pr_CaLT1001M_25C',
        'CV_0015_CS2E0066_Pr_CaLT1001M-1_45C',
        'CV_0015_CS2E0067_Pr_CaLT1001M_45C',
        'CV_0015_CS2E0038_Pr_CaLT1001M_60C',
        'CV_0015_CS2E0042_Pr_CaLT1001M_60C'
    ],
   
}
 
# --- Optional cell labels ---
# If empty ([]), labels are auto-derived from the first 17 characters
# of each cell's ref_month test name (or "Cell_<n>" if that entry is None).
cell_labels = []
 
# --- Check-up cycle number ---
# Cycle number within each monthly check-up test that contains the RPT.
# Typically cycle 2 in calendar aging protocols.
checkup_cycle = 2
 
# Cycle number used for retained capacity/energy extraction.
# This is the first discharge after storage (before any recharge),
# typically cycle 1 in calendar aging protocols.
retained_cycle = 1
 
# Cycle to use for retained capacity/energy on the ref_month (0M) test only.
# In 100% SOC calendar aging protocols the 0M test contains one full
# charge and one full discharge in cycle 2, so we point to cycle 2 here.
# Set to None to use retained_cycle for all months including ref_month.
ref_month_retained_cycle = 2
 
# --- Reference month for relative metrics ---
# SOH, relative energy, relative RTE, relative ΔV all normalise to this month.
ref_month = "0M"
 
# --- DCIR configuration ---
# Set dcir_enabled = False to skip DCIR analysis entirely.
dcir_enabled  = True
test_current  = 42.5            # Amps — pulse current used for R = ΔV / I
target_times  = [0, 1, 10, 60]  # seconds — voltage sample times
 
# Cycle number within each monthly test that contains the DCIR pulses.
# This can differ from checkup_cycle — e.g. set dcir_cycle = 4 if the
# DCIR pulses run on cycle 4 while the RPT capacity check is cycle 2.
dcir_cycle = 4
 
# DCIR step configuration — must match your RPT protocol.
# The pulses are assumed to occur within dcir_cycle of each monthly test.
discharge_steps = [17, 21, 28, 35, 42, 49, 56, 63, 70, 77, 84, 91, 98]
charge_steps    = [17, 24, 31, 38, 45, 52, 59, 66, 73, 80, 87, 94, 98]
 
step_pairs_Dis = {
    '100SOC': (16, 17), '95SOC': (20, 21), '90SOC': (27, 28),
    '80SOC':  (34, 35), '70SOC': (41, 42), '60SOC': (48, 49),
    '50SOC':  (55, 56), '40SOC': (62, 63), '30SOC': (69, 70),
    '20SOC':  (76, 77), '10SOC': (83, 84), '5SOC':  (90, 91),
    '0SOC':   (97, 98),
}
step_pairs_Char = {
    '100SOC': (16, 17), '95SOC': (23, 24), '90SOC': (30, 31),
    '80SOC':  (37, 38), '70SOC': (44, 45), '60SOC': (51, 52),
    '50SOC':  (58, 59), '40SOC': (65, 66), '30SOC': (72, 73),
    '20SOC':  (79, 80), '10SOC': (86, 87), '5SOC':  (93, 94),
    '0SOC':   (97, 98),
}
 
# --- Thickness channel ---
# Voltaiq trace key for the thickness sensor. Set to None to skip.
thickness_channel = 'aux_neware_xls_thickness_mm_0'
 
# --- Output file ---
output_filename = '20260925_CS2E_calendar_aging_results_100SOC_to1M_v3.xlsx'
 

In [17]:
# ================================================================
# CELL 3 — UTILITY & ANALYSIS FUNCTIONS
# Run once to define all functions before executing analyses.
# ================================================================
 
def filter_test_record_by_name(name_fragment, trs):
    """Case-insensitive substring match against all loaded test records."""
    return [t for t in trs if name_fragment.lower() in t.name.lower()]
 
 
def month_to_numeric(month_label):
    """Convert '3M' or '12M' to a float (3.0, 12.0). Returns None if unrecognised."""
    m = re.match(r'^(\d+(?:\.\d+)?)M$', str(month_label).strip())
    return float(m.group(1)) if m else None
 
 
# ----------------------------------------------------------------
# DATE TRACE & CYCLE INFO
# Loads absolute timestamps and cycle counts from one test record.
# Used to compute cumulative time (bridging inter-test gaps) and to
# accumulate cycle counts across monthly check-ups.
# ----------------------------------------------------------------
def get_date_and_cycle_info(tr, checkup_cycle):
    """
    Load the date trace and cycle data from a single Voltaiq test record.
 
    Parameters
    ----------
    tr             : Voltaiq test record (or None)
    checkup_cycle  : int — cycle number of the RPT check-up
 
    Returns
    -------
    start_date              : datetime | None — first timestamp in the test
    checkup_end_date        : datetime | None — last timestamp within checkup_cycle
    max_cycle               : int | None — highest cycle number reached in the test
    test_end_date           : datetime | None — last timestamp in the entire test
    final_cycle_charge_cap  : float | None — charge capacity of the last cycle
    """
    if tr is None:
        return None, None, None, None, None
 
    def _parse_dates(series):
        """Try string parse first; fall back to Excel serial number."""
        parsed = pd.to_datetime(series, errors='coerce')
        if parsed.isna().all():
            try:
                parsed = pd.to_datetime(
                    series.astype(float),
                    origin='1899-12-30', unit='D', errors='coerce',
                )
            except Exception:
                pass
        return parsed
 
    # Attempt 1: load date trace, h_cycle, and h_charge_capacity.
    try:
        reader = tr.make_time_series_reader()
        reader.add_trace_keys(
            'aux_neware_xls_date_none_0', 'h_cycle', 'h_charge_capacity'
        )
        df = reader.read_pandas()
 
        max_cycle = (int(df['h_cycle'].dropna().max())
                     if not df['h_cycle'].dropna().empty else None)
 
        # Final cycle charge capacity (for self-discharge calculation).
        final_cycle_charge_cap = None
        if max_cycle is not None and 'h_charge_capacity' in df.columns:
            df_last = df[df['h_cycle'] == max_cycle]
            v = df_last['h_charge_capacity'].max()
            final_cycle_charge_cap = float(v) if pd.notna(v) else None
 
        df['_dt'] = _parse_dates(df['aux_neware_xls_date_none_0'])
        df_valid  = df.dropna(subset=['_dt'])
 
        if df_valid.empty:
            # Date trace present but unparseable — still return what we have.
            return None, None, max_cycle, None, final_cycle_charge_cap
 
        start_date       = df_valid['_dt'].min()
        test_end_date    = df_valid['_dt'].max()
        df_cyc           = df_valid[df_valid['h_cycle'] == checkup_cycle]
        checkup_end_date = df_cyc['_dt'].max() if not df_cyc.empty else None
 
        return start_date, checkup_end_date, max_cycle, test_end_date, final_cycle_charge_cap
 
    except Exception:
        pass  # date trace key unavailable — fall through
 
    # Attempt 2: date trace unavailable; at least recover max_cycle.
    try:
        reader = tr.make_time_series_reader()
        reader.add_trace_keys('h_cycle')
        df = reader.read_pandas()
        max_cycle = (int(df['h_cycle'].dropna().max())
                     if not df['h_cycle'].dropna().empty else None)
        print(f"      ⚠️ No date trace for {tr.name} — "
              f"cumulative time will be NaN for this check-up.")
        return None, None, max_cycle, None, None
 
    except Exception as e:
        print(f"      ⚠️ Could not read cycle data for {tr.name}: {e}")
        return None, None, None, None, None
 
 
# ----------------------------------------------------------------
# CHECK-UP METRICS EXTRACTION
# Pulls all RPT metrics at the check-up cycle from one test record.
# ----------------------------------------------------------------
def extract_checkup_metrics(tr, checkup_cycle, retained_cycle=1,
                             retained_override_cycle=None,
                             thickness_channel=None):
    """
    Extract RPT metrics at checkup_cycle (and retained_cycle) from one
    Voltaiq test record.
 
    Parameters
    ----------
    tr              : Voltaiq test record (or None for a missing check-up)
    checkup_cycle   : int — recovered capacity/energy cycle
    retained_cycle  : int — first discharge after storage (retained metrics)
 
    Returns
    -------
    dict of metric_name → value (np.nan for any missing value).
    """
    _nan = {
        'Test Name':                              None,
        'Retained Discharge Capacity (Ah)':       np.nan,
        'Retained Discharge Energy (Wh)':         np.nan,
        'Recovered Charge Capacity (Ah)':         np.nan,
        'Recovered Discharge Capacity (Ah)':      np.nan,
        'Coulombic Efficiency':                   np.nan,
        'Recovered Charge Energy (Wh)':           np.nan,
        'Recovered Discharge Energy (Wh)':        np.nan,
        'Energy Efficiency':                      np.nan,
        'Energy Inefficiency':                    np.nan,
        'Capacity Weighted deltaV (mV)':          np.nan,
        'Time Weighted deltaV (mV)':              np.nan,
        'Max Charge Temperature (°C)':            np.nan,
        'Max Discharge Temperature (°C)':         np.nan,
        'Ambient Temperature (°C)':               np.nan,
        'Thickness (mm)':                         np.nan,
    }
 
    if tr is None:
        return _nan
 
    print(f"    🔍 {tr.name}")
 
    base_keys = [
        'h_cycle', 'h_test_time',
        'h_charge_capacity', 'h_discharge_capacity',
        'h_charge_energy', 'h_discharge_energy',
        'h_potential',
    ]
 
    # Try to load both temperature channels; fall back gracefully through
    # progressively fewer channels until base keys alone succeed.
    loaded_keys = base_keys
    for attempt in [
        base_keys + ['aux_neware_xls_t1_none_0', 'aux_neware_xls_t2_none_0'],
        base_keys + ['aux_neware_xls_t2_none_0'],
        base_keys + ['aux_neware_xls_t1_none_0'],
        base_keys,
    ]:
        try:
            reader = tr.make_time_series_reader()
            reader.add_trace_keys(*attempt)
            loaded_keys = attempt
            break
        except Exception:
            continue
    else:
        print(f"      ⚠️ Could not load any trace keys for {tr.name}.")
        result = _nan.copy()
        result['Test Name'] = tr.name
        return result
 
    has_t1 = 'aux_neware_xls_t1_none_0' in loaded_keys
    has_t2 = 'aux_neware_xls_t2_none_0' in loaded_keys
    if not has_t1 and not has_t2:
        print(f"      ⚠️ No temperature data available for {tr.name}.")
 
    df     = reader.read_pandas().dropna(subset=['h_cycle'])
    df_cyc = df[df['h_cycle'] == checkup_cycle]
 
    if df_cyc.empty:
        print(f"      ⚠️ Checkup cycle {checkup_cycle} not found in {tr.name} "
              f"— metrics set to NaN.")
        result = _nan.copy()
        result['Test Name'] = tr.name
        return result
 
    # ── Retained metrics ─────────────────────────────────────────────────
    # If an override cycle is specified (used for the ref_month test when
    # its protocol differs), extract from that cycle; otherwise use retained_cycle.
    _ret_cycle = retained_override_cycle if retained_override_cycle is not None else retained_cycle
    df_ret = df[df['h_cycle'] == _ret_cycle]
    if not df_ret.empty:
        retained_discharge_cap    = df_ret['h_discharge_capacity'].max()
        retained_discharge_energy = df_ret['h_discharge_energy'].max()
    else:
        print(f"      ⚠️ Retained cycle {_ret_cycle} not found in {tr.name}.")
        retained_discharge_cap    = np.nan
        retained_discharge_energy = np.nan
 
    # ── Recovered capacity (checkup_cycle) ───────────────────────
    charge_cap    = df_cyc['h_charge_capacity'].max()
    discharge_cap = df_cyc['h_discharge_capacity'].max()
    coulombic_eff = (
        discharge_cap / charge_cap
        if (pd.notna(charge_cap) and charge_cap > 0 and pd.notna(discharge_cap))
        else np.nan
    )
 
    # ── Recovered energy ─────────────────────────────────────────
    charge_energy    = df_cyc['h_charge_energy'].max()
    discharge_energy = df_cyc['h_discharge_energy'].max()
 
    if (pd.notna(charge_energy) and charge_energy > 0
            and pd.notna(discharge_energy)):
        rte_frac     = discharge_energy / charge_energy
        rte          = rte_frac
        energy_ineff = 1 - rte_frac
    else:
        rte          = np.nan
        energy_ineff = np.nan
 
    # ── Phase separation for temperature and ΔV ──────────────────
    df_chg = (df_cyc[df_cyc['h_charge_capacity'] > 0]
              .dropna(subset=['h_potential', 'h_test_time', 'h_charge_capacity']))
    df_dis = (df_cyc[df_cyc['h_discharge_capacity'] > 0]
              .dropna(subset=['h_potential', 'h_test_time', 'h_discharge_capacity']))
 
    # ── Temperatures ─────────────────────────────────────────────
    t2_col = 'aux_neware_xls_t2_none_0'
    max_charge_temp = max_discharge_temp = np.nan
    if has_t2 and t2_col in df.columns:
        max_charge_temp    = df_chg[t2_col].max() if not df_chg.empty else np.nan
        max_discharge_temp = df_dis[t2_col].max() if not df_dis.empty else np.nan
 
    t1_col = 'aux_neware_xls_t1_none_0'
    ambient_temp = np.nan
    if has_t1 and t1_col in df.columns:
        ambient_temp = df_cyc[t1_col].mean()
 
    # ── Thickness ─────────────────────────────────────────────────────
    # Loaded in a separate reader so a missing channel doesn't affect
    # the main data load.
    thickness = np.nan
    if thickness_channel:
        try:
            _tk_reader = tr.make_time_series_reader()
            _tk_reader.add_trace_keys(thickness_channel, 'h_cycle')
            _df_tk     = _tk_reader.read_pandas().dropna(subset=['h_cycle'])
            _df_tk_cyc = _df_tk[_df_tk['h_cycle'] == checkup_cycle]
            if not _df_tk_cyc.empty:
                thickness = _df_tk_cyc[thickness_channel].mean()
        except Exception:
            pass
 
    # ── ΔV calculations ───────────────────────────────────────────
    cap_dv = time_dv = np.nan
 
    # Capacity-weighted ΔV: sort by capacity (monotonic during charge/discharge).
    # Drop duplicate capacity values to prevent scipy interpolation errors.
    df_chg_s = (df_chg.sort_values('h_charge_capacity')
                .drop_duplicates(subset=['h_charge_capacity']))
    df_dis_s = (df_dis.sort_values('h_discharge_capacity')
                .drop_duplicates(subset=['h_discharge_capacity']))
 
    if len(df_chg_s) > 1 and len(df_dis_s) > 1:
        q_chg = df_chg_s['h_charge_capacity'].values
        v_chg = df_chg_s['h_potential'].values
        q_dis = df_dis_s['h_discharge_capacity'].values
        v_dis = df_dis_s['h_potential'].values
 
        # Capacity-weighted ΔV: interpolate both curves onto a common
        # capacity grid, integrate (V_chg - V_dis), divide by range.
        q_min = max(q_chg.min(), q_dis.min())
        q_max = min(q_chg.max(), q_dis.max())
        if q_max > q_min:
            q_common = np.linspace(q_min, q_max, 500)
            f_chg_q  = interpolate.interp1d(q_chg, v_chg,
                           bounds_error=False, fill_value='extrapolate')
            f_dis_q  = interpolate.interp1d(q_dis, v_dis,
                           bounds_error=False, fill_value='extrapolate')
            dv_q     = f_chg_q(q_common) - f_dis_q(q_common)
            cap_dv   = np.trapz(dv_q, q_common) / (q_max - q_min) * 1000  # V → mV
 
    # Time-weighted ΔV: sort by TIME (not capacity) so that np.trapz
    # always receives a monotonically increasing x-axis. Calendar aging
    # protocols can have CV holds, rest steps, and variable C-rates that
    # make capacity-sorted time values non-monotonic, causing sign errors.
    df_chg_t = (df_chg.sort_values('h_test_time')
                .drop_duplicates(subset=['h_test_time']))
    df_dis_t = (df_dis.sort_values('h_test_time')
                .drop_duplicates(subset=['h_test_time']))
 
    if len(df_chg_t) > 1 and len(df_dis_t) > 1:
        t_chg  = df_chg_t['h_test_time'].values
        v_chg_t = df_chg_t['h_potential'].values
        t_dis  = df_dis_t['h_test_time'].values
        v_dis_t = df_dis_t['h_potential'].values
        rng_c  = t_chg.max() - t_chg.min()
        rng_d  = t_dis.max() - t_dis.min()
        if rng_c > 0 and rng_d > 0:
            v_chg_mean = np.trapz(v_chg_t, t_chg) / rng_c
            v_dis_mean = np.trapz(v_dis_t, t_dis) / rng_d
            time_dv    = (v_chg_mean - v_dis_mean) * 1000  # V → mV
 
    return {
        'Test Name':                              tr.name,
        'Retained Discharge Capacity (Ah)':       retained_discharge_cap,
        'Retained Discharge Energy (Wh)':         retained_discharge_energy,
        'Recovered Charge Capacity (Ah)':         charge_cap,
        'Recovered Discharge Capacity (Ah)':      discharge_cap,
        'Coulombic Efficiency':                   coulombic_eff,
        'Recovered Charge Energy (Wh)':           charge_energy,
        'Recovered Discharge Energy (Wh)':        discharge_energy,
        'Energy Efficiency':                      rte,
        'Energy Inefficiency':                    energy_ineff,
        'Capacity Weighted deltaV (mV)':          cap_dv,
        'Time Weighted deltaV (mV)':              time_dv,
        'Max Charge Temperature (°C)':            max_charge_temp,
        'Max Discharge Temperature (°C)':         max_discharge_temp,
        'Ambient Temperature (°C)':               ambient_temp,
        'Thickness (mm)':                         thickness,
    }
 
 
# ----------------------------------------------------------------
# CALENDAR AGING SERIES
# Processes one cell across all months, then adds relative metrics.
# ----------------------------------------------------------------
def calculate_calendar_aging_series(
    cell_position,
    cell_label,
    filtered_trs_by_month,
    checkup_cycle,
    retained_cycle,
    ref_month,
    ref_month_retained_cycle=None,
    thickness_channel=None,
):
    """
    Calculate calendar aging metrics for one cell across all months.
 
    Parameters
    ----------
    cell_position           : int — index into each month's test list
    cell_label              : str — display name for this cell
    filtered_trs_by_month   : dict[str, list] — month → test record list
    checkup_cycle           : int — RPT cycle to extract metrics from
    ref_month               : str — reference month for relative metrics
 
    Returns
    -------
    pd.DataFrame with one row per month, all absolute and relative metrics.
    Columns include:
      • Cumulative Time (days) — absolute calendar days since start of ref_month
        test, bridging inter-test gaps via aux_neware_xls_date_none_0.
      • Cumulative Cycle — total RPT cycles run across all months up to and
        including checkup_cycle of the current month.
    """
    print(f"\n📋 Cell [{cell_position}]: {cell_label}")
 
    # Sort months chronologically so cumulative counters accumulate correctly.
    months_sorted = sorted(
        filtered_trs_by_month.keys(),
        key=lambda x: (month_to_numeric(x) if month_to_numeric(x) is not None
                       else float('inf'))
    )
 
    # ── Step 1: Collect date & cycle info for every month ─────────
    print(f"  📅 Loading date traces ...")
    date_info = {}  # month → (start_date, checkup_end_date, max_cycle, test_end_date, final_cycle_charge_cap)
    for month in months_sorted:
        trs_list       = filtered_trs_by_month[month]
        tr             = trs_list[cell_position] if cell_position < len(trs_list) else None
        date_info[month] = get_date_and_cycle_info(tr, checkup_cycle)
 
    # ── Step 2: Global time origin = first timestamp of ref_month test ──
    t0_global = date_info.get(ref_month, (None,)*5)[0]
    if t0_global is None:
        print(f"  ⚠️ No date trace found in '{ref_month}' for {cell_label} — "
              f"'Cumulative Time (days)' will be NaN (inter-test gaps cannot be measured).")
 
    # ── Step 3: Per-month cumulative cycle offsets ────────────────
    # offset[month] = total cycles completed in all months BEFORE this one.
    # After each month, add that month's max_cycle to the running total so
    # cycles from every RPT stack on top of each other.
    cumulative_offset    = 0
    month_cycle_offsets  = {}
    for month in months_sorted:
        month_cycle_offsets[month] = cumulative_offset
        _, _, max_cyc, _, _ = date_info[month]
        if max_cyc is not None:
            cumulative_offset += max_cyc
 
    # ── Step 4: Extract RPT metrics and attach time / cycle columns ──
    rows = []
    for month in months_sorted:
        trs_list = filtered_trs_by_month[month]
        tr       = trs_list[cell_position] if cell_position < len(trs_list) else None
        # For the reference month, optionally redirect retained extraction
        # to a different cycle (e.g. cycle 2 for 100SOC 0M tests).
        _override = ref_month_retained_cycle if month == ref_month else None
        m = extract_checkup_metrics(tr, checkup_cycle, retained_cycle,
                                    retained_override_cycle=_override,
                                    thickness_channel=thickness_channel)
 
        # Cumulative time: absolute calendar days from t0_global to the
        # last data point within checkup_cycle of this month's test.
        _, checkup_end, _, _, _ = date_info[month]
        if t0_global is not None and checkup_end is not None:
            cumulative_time_days = (
                (checkup_end - t0_global).total_seconds() / 86400
            )
        else:
            cumulative_time_days = np.nan
 
        # Cumulative cycle: prior months' total + this month's checkup cycle.
        offset           = month_cycle_offsets[month]
        cumulative_cycle = (offset + checkup_cycle) if tr is not None else np.nan
 
        m['Month']                  = month
        m['Calendar Time (months)'] = month_to_numeric(month)
        m['Time (days)']            = cumulative_time_days
        m['Cycle']                  = cumulative_cycle
        m['Cell Label']             = cell_label
        rows.append(m)
 
    df = (pd.DataFrame(rows)
          .sort_values('Calendar Time (months)')
          .reset_index(drop=True))
 
    # ── Step 5: Relative metrics normalised to ref_month ─────────
    ref_mask = df['Month'] == ref_month
    if not ref_mask.any():
        print(f"  ⚠️ Reference month '{ref_month}' not found for {cell_label} "
              f"— relative metrics will be NaN.")
        ref = {}
    else:
        r   = df[ref_mask].iloc[0]
        ref = {
            'recovered_discharge_cap':    r['Recovered Discharge Capacity (Ah)'],
            'retained_discharge_cap':     r['Retained Discharge Capacity (Ah)'],
            'recovered_discharge_energy': r['Recovered Discharge Energy (Wh)'],
            'retained_discharge_energy':  r['Retained Discharge Energy (Wh)'],
            'energy_eff':                 r['Energy Efficiency'],
            'energy_ineff':               r['Energy Inefficiency'],
            'cap_dv':                     r['Capacity Weighted deltaV (mV)'],
            'time_dv':                    r['Time Weighted deltaV (mV)'],
            'thickness':                  r['Thickness (mm)'],
        }
 
    def _rel(val, key):
        rv = ref.get(key, np.nan)
        return (val / rv
                if (pd.notna(val) and pd.notna(rv)
                    and rv != 0 and not np.isinf(rv))
                else np.nan)
 
    df['Relative Discharge Capacity']        = df['Recovered Discharge Capacity (Ah)'].apply(lambda x: _rel(x, 'recovered_discharge_cap'))
    df['Relative Retained Discharge Capacity'] = df['Retained Discharge Capacity (Ah)'].apply(lambda x: _rel(x, 'retained_discharge_cap'))
    df['Relative Discharge Energy']          = df['Recovered Discharge Energy (Wh)'].apply(lambda x: _rel(x, 'recovered_discharge_energy'))
    df['Relative Retained Discharge Energy'] = df['Retained Discharge Energy (Wh)'].apply(lambda x: _rel(x, 'retained_discharge_energy'))
    df['Relative Energy Efficiency']         = df['Energy Efficiency'].apply(lambda x: _rel(x, 'energy_eff'))
    df['Relative Energy Inefficiency']       = df['Energy Inefficiency'].apply(lambda x: _rel(x, 'energy_ineff'))
    df['Relative Capacity Weighted deltaV']  = df['Capacity Weighted deltaV (mV)'].apply(lambda x: _rel(x, 'cap_dv'))
    df['Relative Time Weighted deltaV']      = df['Time Weighted deltaV (mV)'].apply(lambda x: _rel(x, 'time_dv'))
    df['Relative Thickness']                 = df['Thickness (mm)'].apply(lambda x: _rel(x, 'thickness'))
 
    # ── Step 6: Self-discharge (Ah) ───────────────────────────────
    # For each month look back to the last available month with data.
    # Formula: [(final_charge_cap_prev - retained_cap_curr)
    #           - (recovered_cap_curr - recovered_cap_prev)] / elapsed_hours
    months_with_data = []
    for _m in months_sorted:
        _trs = filtered_trs_by_month.get(_m, [])
        _tr  = _trs[cell_position] if cell_position < len(_trs) else None
        if _tr is not None:
            months_with_data.append(_m)
 
    prev_month_map = {months_with_data[j]: months_with_data[j - 1]
                      for j in range(1, len(months_with_data))}
 
    month_to_row = df.set_index('Month').to_dict('index')
 
    self_discharges = []
    for _month in df['Month']:
        if _month not in prev_month_map:
            self_discharges.append(np.nan)
            continue
 
        _prev = prev_month_map[_month]
        _curr_row = month_to_row.get(_month, {})
        _prev_row = month_to_row.get(_prev, {})
 
        _, _, _, _test_end_prev, _final_charge_cap_prev = date_info[_prev]
        _start_curr, _, _, _, _                         = date_info[_month]
 
        if _test_end_prev is None or _start_curr is None:
            self_discharges.append(np.nan)
            continue
 
        _elapsed_hours = (_start_curr - _test_end_prev).total_seconds() / 3600
        if _elapsed_hours <= 0:
            self_discharges.append(np.nan)
            continue
 
        _ret_cap_curr  = _curr_row.get('Retained Discharge Capacity (Ah)',  np.nan)
        _rec_cap_curr  = _curr_row.get('Recovered Discharge Capacity (Ah)', np.nan)
        _rec_cap_prev  = _prev_row.get('Recovered Discharge Capacity (Ah)', np.nan)
 
        if any(pd.isna(v) for v in [_final_charge_cap_prev, _ret_cap_curr,
                                     _rec_cap_curr, _rec_cap_prev]):
            self_discharges.append(np.nan)
            continue
 
        _term1 = _final_charge_cap_prev - _ret_cap_curr
        _term2 = _rec_cap_curr - _rec_cap_prev
        self_discharges.append((_term1 - _term2) / _elapsed_hours)
 
    df['Self Discharge (Ah)'] = self_discharges
 
    # ── Step 7: Ensure ref_month relative metrics are exactly 1 ──────────
    # When an absolute metric is NaN at the reference month (e.g. missing
    # temperature channel, retained cycle not found), the NaN/NaN division
    # produces NaN instead of 1. Explicitly set all relative columns to 1
    # for the reference row so the baseline is always unambiguous.
    relative_cols = [
        'Relative Retained Discharge Capacity',
        'Relative Discharge Capacity',
        'Relative Retained Discharge Energy',
        'Relative Discharge Energy',
        'Relative Energy Efficiency',
        'Relative Energy Inefficiency',
        'Relative Capacity Weighted deltaV',
        'Relative Time Weighted deltaV',
        'Relative Thickness',
    ]
    ref_idx = df.index[df['Month'] == ref_month]
    for col in relative_cols:
        if col in df.columns:
            df.loc[ref_idx, col] = 1
 
    return df
 
 
# ----------------------------------------------------------------
# DCIR PIPELINE
# Identical logic to the cycle life master code; for calendar aging,
# pass one test record per month (for one cell) and set
# cycle_step_list to [(checkup_cycle, step), ...].
# ----------------------------------------------------------------
def run_dcir_pipeline(
    filtered_trs,
    cycle_step_list_charge,
    cycle_step_list_discharge,
    target_times,
    step_pairs_Dis,
    step_pairs_Char,
    test_current,
):
    """
    Run the full DCIR pipeline for a list of Voltaiq test records.
 
    Parameters
    ----------
    filtered_trs              : list — Voltaiq test records (Nones filtered out)
    cycle_step_list_charge    : list of (cycle, step) tuples — charge pulses
    cycle_step_list_discharge : list of (cycle, step) tuples — discharge pulses
    target_times              : list[int] — seconds at which to sample voltage
    step_pairs_Dis            : dict — SOC label → (pre-pulse step, pulse step)
    step_pairs_Char           : dict — SOC label → (pre-pulse step, pulse step)
    test_current              : float — pulse current in Amps
 
    Returns
    -------
    pd.DataFrame (empty if no DCIR data found).
    """
    time_lookup = {1: "00:00:01", 10: "00:00:10", 60: "00:01:00"}
    all_results = []
 
    for tr in filtered_trs:
        if tr is None:
            continue
        print(f"  🔍 DCIR: {tr.name}")
        try:
            reader     = tr.make_time_series_reader()
            trace_keys = [
                'aux_neware_xls_voltage_volt_0',
                'h_cycle',
                'h_step_index',
                'aux_neware_xls_time_none_0',
            ]
            try:
                reader.add_trace_keys(*trace_keys)
            except Exception as ve:
                print(f"    ⚠️ Could not add DCIR trace keys: {ve}")
                try:
                    reader = tr.make_time_series_reader()
                    reader.add_trace_keys(*[k for k in trace_keys
                                            if 'current' not in k])
                except Exception as ve2:
                    print(f"    ❌ Skipping {tr.name}: {ve2}")
                    continue
            df = reader.read_pandas()
        except Exception as e:
            print(f"    ❌ Skipping {tr.name}: {e}")
            continue
 
        df['aux_neware_xls_time_none_0'] = (
            df['aux_neware_xls_time_none_0'].astype(str)
        )
        df_sorted = df.sort_values(
            ['h_cycle', 'h_step_index', 'aux_neware_xls_time_none_0']
        )
 
        for mode, cycle_step_list in [
            ("charge",    cycle_step_list_charge),
            ("discharge", cycle_step_list_discharge),
        ]:
            for cycle, step in cycle_step_list:
                for t in target_times:
                    if t == 0:
                        prev = df_sorted[
                            (df_sorted['h_cycle']      == cycle) &
                            (df_sorted['h_step_index'] == step - 1)
                        ]
                        if not prev.empty:
                            all_results.append({
                                "test_name":   tr.name,
                                "mode":        mode,
                                "cycle":       cycle,
                                "step":        step - 1,
                                "time_s":      0,
                                "potential_V": prev.iloc[-1][
                                    'aux_neware_xls_voltage_volt_0'],
                            })
                    else:
                        step_data = df_sorted[
                            (df_sorted['h_cycle']                    == cycle) &
                            (df_sorted['h_step_index']               == step) &
                            (df_sorted['aux_neware_xls_time_none_0'] == time_lookup[t])
                        ]
                        if not step_data.empty:
                            all_results.append({
                                "test_name":   tr.name,
                                "mode":        mode,
                                "cycle":       cycle,
                                "step":        step,
                                "time_s":      t,
                                "potential_V": step_data.iloc[0][
                                    'aux_neware_xls_voltage_volt_0'],
                            })
 
    if not all_results:
        print("  ⚠️ No DCIR voltage data extracted — check step configuration.")
        return pd.DataFrame()
 
    df_pot = pd.DataFrame(all_results)
    pivot  = df_pot.pivot_table(
        index=['test_name', 'cycle', 'step'],
        columns='time_s', values='potential_V',
    ).reset_index()
    pivot.columns.name = None
    pivot = pivot.rename(columns={0: 'V_0s', 1: 'V_1s', 10: 'V_10s', 60: 'V_60s'})
 
    def process_pairs(step_pairs, mode_label):
        parts = {}
        for soc_label, (s0, s1) in step_pairs.items():
            df0 = pivot[pivot['step'] == s0][['test_name', 'cycle', 'V_0s']]
            df1 = pivot[pivot['step'] == s1][
                ['test_name', 'cycle', 'V_1s', 'V_10s', 'V_60s']
            ]
            merged               = pd.merge(df0, df1, on=['test_name', 'cycle'])
            merged['SOC']        = int(soc_label.replace('SOC', ''))
            merged['SOC_label']  = soc_label
            merged['Charge/Discharge'] = mode_label
            parts[soc_label]     = merged
        return parts
 
    def add_deltas(parts_dict):
        frames = []
        for _, d in parts_dict.items():
            d = d.copy()
            d['ΔV_1s']  = (d['V_0s'] - d['V_1s']).abs()
            d['ΔV_10s'] = (d['V_0s'] - d['V_10s']).abs()
            d['ΔV_60s'] = (d['V_0s'] - d['V_60s']).abs()
            frames.append(d)
        return pd.concat(frames, ignore_index=True)
 
    combined = pd.concat(
        [add_deltas(process_pairs(step_pairs_Dis,  'Discharge')),
         add_deltas(process_pairs(step_pairs_Char, 'Charge'))],
        ignore_index=True,
    )
    combined['R_1s (mΩ)']  = (combined['ΔV_1s']  / test_current) * 1000
    combined['R_10s (mΩ)'] = (combined['ΔV_10s'] / test_current) * 1000
    combined['R_60s (mΩ)'] = (combined['ΔV_60s'] / test_current) * 1000
    combined['SOC'] = (
        combined['SOC_label'].str.replace('SOC', '', regex=False).astype(float)
    )
    sort_cols = [c for c in ['test_name', 'cycle', 'Charge/Discharge', 'SOC']
                 if c in combined.columns]
    return combined.sort_values(by=sort_cols).reset_index(drop=True)
 
 
# ----------------------------------------------------------------
# DCIR SUMMARY METRICS — calendar aging version
# Normalises relative metrics by the ref_month check-up, not a cycle.
# ----------------------------------------------------------------
def calculate_dcir_summary_metrics_cal(combined_all, ref_month_dcir, trs_to_month_map):
    """
    Compute DCIR summary metrics for calendar aging data.
 
    Parameters
    ----------
    combined_all       : pd.DataFrame — output of run_dcir_pipeline()
    ref_month_dcir     : str — reference month for relative metrics (e.g. "0M")
    trs_to_month_map   : dict[str, str] — test_name → month label
 
    Returns
    -------
    pd.DataFrame with one row per (test_name, Month, Charge/Discharge).
    """
    combined_all          = combined_all.copy()
    combined_all['Month'] = combined_all['test_name'].map(trs_to_month_map)
 
    results = []
    for (test_name, month, mode), grp in combined_all.groupby(
        ['test_name', 'Month', 'Charge/Discharge']
    ):
        row_50    = grp[grp['SOC'] == 50]
        dcr_50soc = row_50['R_10s (mΩ)'].values[0] if not row_50.empty else None
 
        valid = grp[['SOC', 'R_10s (mΩ)']].dropna()
        soc_weighted_dcr = (
            np.average(valid['R_10s (mΩ)'], weights=valid['SOC'])
            if (not valid.empty and valid['SOC'].sum() > 0) else None
        )
        results.append({
            'test_name':                  test_name,
            'Month':                      month,
            'Charge/Discharge':           mode,
            '10s DCR 50%SOC (mΩ)':       dcr_50soc,
            'SOC Weighted 10s DCR (mΩ)': soc_weighted_dcr,
        })
 
    dcir_summary = pd.DataFrame(results)
 
    # Relative DCIR metrics are computed in Cell 7 after the merge into
    # cal_aging_summary, where Cell Label and Month are unambiguous.
    return dcir_summary

In [18]:
# ================================================================
# CELL 4 — DATA LOADING
# ================================================================
 
n_cells = max(len(v) for v in test_names_by_month.values())
print(f"ℹ️  Detected {n_cells} cell position(s) across "
      f"{len(test_names_by_month)} month(s).")
 
trs_all = vs.get_test_records()
 
# Build filtered_trs_by_month preserving positional alignment.
filtered_trs_by_month = {}
for month, names in test_names_by_month.items():
    month_trs = []
    for name in names:
        if name is None:
            month_trs.append(None)
        else:
            matched = filter_test_record_by_name(name, trs_all)
            if matched:
                month_trs.append(matched[0])
            else:
                print(f"  ❌ No match for '{name}' ({month})")
                month_trs.append(None)
    filtered_trs_by_month[month] = month_trs
 
# Build cell labels from ref_month names if not provided by the user.
if not cell_labels:
    ref_trs = filtered_trs_by_month.get(ref_month, [])
    cell_labels = []
    for i in range(n_cells):
        tr = ref_trs[i] if i < len(ref_trs) else None
        cell_labels.append(tr.name[:17] if tr else f"Cell_{i}")
 
print(f"\n✅ Cell labels: {cell_labels}")

ℹ️  Detected 6 cell position(s) across 2 month(s).

✅ Cell labels: ['CV_0015_CS2E0060_', 'CV_0015_CS2E0023_', 'CV_0015_CS2E0066_', 'CV_0015_CS2E0067_', 'CV_0015_CS2E0038_', 'CV_0015_CS2E0042_']


In [19]:
# ================================================================
# CELL 5 — RUN CALENDAR AGING SERIES ANALYSIS
# ================================================================
 
cal_aging_results = []
 
for i, label in enumerate(cell_labels):
    df_cell = calculate_calendar_aging_series(
        cell_position            = i,
        cell_label               = label,
        filtered_trs_by_month    = filtered_trs_by_month,
        checkup_cycle            = checkup_cycle,
        retained_cycle           = retained_cycle,
        ref_month                = ref_month,
        ref_month_retained_cycle = ref_month_retained_cycle,
        thickness_channel        = thickness_channel,
    )
    if not df_cell.empty:
        cal_aging_results.append(df_cell)
 
cal_aging_summary = pd.concat(cal_aging_results, ignore_index=True)
 
print("\n📊 Calendar Aging Summary (preview):")
print(cal_aging_summary[
    ['Cell Label', 'Month', 'Recovered Discharge Capacity (Ah)', 'Relative Discharge Capacity']
].head(12))


📋 Cell [0]: CV_0015_CS2E0060_
  📅 Loading date traces ...
    🔍 CV_0015_CS2E0060_Pr_CaLT1000M_25C
      ⚠️ No temperature data available for CV_0015_CS2E0060_Pr_CaLT1000M_25C.
    🔍 CV_0015_CS2E0060_Pr_CaLT1001M_25C
      ⚠️ No temperature data available for CV_0015_CS2E0060_Pr_CaLT1001M_25C.

📋 Cell [1]: CV_0015_CS2E0023_
  📅 Loading date traces ...
    🔍 CV_0015_CS2E0023_Pr_CaLT1000M_25C
      ⚠️ No temperature data available for CV_0015_CS2E0023_Pr_CaLT1000M_25C.
    🔍 CV_0015_CS2E0023_Pr_CaLT1001M_25C
      ⚠️ No temperature data available for CV_0015_CS2E0023_Pr_CaLT1001M_25C.

📋 Cell [2]: CV_0015_CS2E0066_
  📅 Loading date traces ...
    🔍 CV_0015_CS2E0066_Pr_CaLT1000M_45C
      ⚠️ No temperature data available for CV_0015_CS2E0066_Pr_CaLT1000M_45C.
    🔍 CV_0015_CS2E0066_Pr_CaLT1001M-1_45C
      ⚠️ No temperature data available for CV_0015_CS2E0066_Pr_CaLT1001M-1_45C.

📋 Cell [3]: CV_0015_CS2E0067_
  📅 Loading date traces ...
    🔍 CV_0015_CS2E0067_Pr_CaLT1000M_45C
      ⚠️ No 

In [20]:
# ================================================================
# CELL 6 — RUN DCIR ANALYSIS  (only if dcir_enabled = True)
# ================================================================
 
dcir_raw_all     = pd.DataFrame()
dcir_summary_all = pd.DataFrame()
 
if dcir_enabled:
    # DCIR pulses live inside dcir_cycle of each monthly test.
    cycle_step_dis = [(dcir_cycle, s) for s in discharge_steps]
    cycle_step_chg = [(dcir_cycle, s) for s in charge_steps]
 
    dcir_all_cells = []
    for i, label in enumerate(cell_labels):
        print(f"\n🔬 DCIR — Cell [{i}]: {label}")
 
        # Collect test records for this cell position across all months.
        cell_trs_valid = []
        trs_to_month_map = {}
        for month, trs_list in filtered_trs_by_month.items():
            tr = trs_list[i] if i < len(trs_list) else None
            if tr is not None:
                cell_trs_valid.append(tr)
                trs_to_month_map[tr.name] = month
 
        if not cell_trs_valid:
            print(f"  ⚠️ No test records found — skipping DCIR for {label}.")
            continue
 
        combined = run_dcir_pipeline(
            filtered_trs              = cell_trs_valid,
            cycle_step_list_charge    = cycle_step_chg,
            cycle_step_list_discharge = cycle_step_dis,
            target_times              = target_times,
            step_pairs_Dis            = step_pairs_Dis,
            step_pairs_Char           = step_pairs_Char,
            test_current              = test_current,
        )
 
        if combined.empty:
            print(f"  ⚠️ No DCIR data extracted for {label}.")
            continue
 
        combined['Cell Label'] = label
        dcir_summary_cell = calculate_dcir_summary_metrics_cal(
            combined_all     = combined,
            ref_month_dcir   = ref_month,
            trs_to_month_map = trs_to_month_map,
        )
        dcir_summary_cell['Cell Label'] = label
        dcir_all_cells.append((combined, dcir_summary_cell))
 
    if dcir_all_cells:
        dcir_raw_all     = pd.concat([x[0] for x in dcir_all_cells], ignore_index=True)
        dcir_summary_all = pd.concat([x[1] for x in dcir_all_cells], ignore_index=True)
        print("\n📊 DCIR Summary (preview):")
        print(dcir_summary_all.head())
    else:
        print("  ⚠️ DCIR pipeline returned no data for any cell.")
else:
    print("ℹ️  DCIR analysis skipped (dcir_enabled = False).")


🔬 DCIR — Cell [0]: CV_0015_CS2E0060_
  🔍 DCIR: CV_0015_CS2E0060_Pr_CaLT1000M_25C
  🔍 DCIR: CV_0015_CS2E0060_Pr_CaLT1001M_25C

🔬 DCIR — Cell [1]: CV_0015_CS2E0023_
  🔍 DCIR: CV_0015_CS2E0023_Pr_CaLT1000M_25C
  🔍 DCIR: CV_0015_CS2E0023_Pr_CaLT1001M_25C

🔬 DCIR — Cell [2]: CV_0015_CS2E0066_
  🔍 DCIR: CV_0015_CS2E0066_Pr_CaLT1000M_45C
  🔍 DCIR: CV_0015_CS2E0066_Pr_CaLT1001M-1_45C

🔬 DCIR — Cell [3]: CV_0015_CS2E0067_
  🔍 DCIR: CV_0015_CS2E0067_Pr_CaLT1000M_45C
  🔍 DCIR: CV_0015_CS2E0067_Pr_CaLT1001M_45C

🔬 DCIR — Cell [4]: CV_0015_CS2E0038_
  🔍 DCIR: CV_0015_CS2E0038_Pr_CaLT1000M_60C
  🔍 DCIR: CV_0015_CS2E0038_Pr_CaLT1001M_60C

🔬 DCIR — Cell [5]: CV_0015_CS2E0042_
  🔍 DCIR: CV_0015_CS2E0042_Pr_CaLT1000M_60C
  🔍 DCIR: CV_0015_CS2E0042_Pr_CaLT1001M_60C

📊 DCIR Summary (preview):
                           test_name Month Charge/Discharge  \
0  CV_0015_CS2E0060_Pr_CaLT1000M_25C    0M           Charge   
1  CV_0015_CS2E0060_Pr_CaLT1000M_25C    0M        Discharge   
2  CV_0015_CS2E0060_Pr_CaL

In [21]:
# ================================================================
# CELL 7 — EXPORT ALL RESULTS TO EXCEL
# Output structure:
#   • One sheet per cell — one row per month, missing values as "NAN".
#   • "SOH Summary" sheet — wide-format table (cells × months).
#   • "DCIR Raw" sheet — only if dcir_enabled and data available.
#
# Column names are mapped explicitly: LEFT side = internal column name
# created in Cells 3/5 (must match exactly), RIGHT side = header written
# to Excel (edit freely). Any unmatched name is reported, never dropped
# silently. Safe to re-run: works on a copy of cal_aging_summary.
# ================================================================

print(f"\n💾 Preparing export to '{output_filename}' ...")

export_df = cal_aging_summary.copy()

# --- Main metrics: internal name → Excel header ---
column_map = {
    'Month':                                'Month',
    'Time (days)':                          'Time (days)',
    'Cycle':                                'Cycle',
    'Retained Discharge Capacity (Ah)':     'Retained discharge capacity (Ah)',
    'Recovered Charge Capacity (Ah)':       'Recovered charge capacity (Ah)',
    'Recovered Discharge Capacity (Ah)':    'Recovered discharge capacity (Ah)',
    'Relative Retained Discharge Capacity': 'Relative retained discharge capacity',
    'Relative Discharge Capacity':          'Relative discharge capacity',
    'Self Discharge (Ah)':                  'Self discharge (Ah)',
    'Coulombic Efficiency':                 'Coulombic efficiency',
    'Retained Discharge Energy (Wh)':       'Retained discharge energy (Wh)',
    'Recovered Charge Energy (Wh)':         'Recovered charge energy (Wh)',
    'Recovered Discharge Energy (Wh)':      'Recovered discharge energy (Wh)',
    'Relative Retained Discharge Energy':   'Relative retained discharge energy',
    'Relative Discharge Energy':            'Relative discharge energy',
    'Energy Efficiency':                    'Energy efficiency',
    'Energy Inefficiency':                  'Energy inefficiency',
    'Relative Energy Efficiency':           'Relative energy efficiency',
    'Relative Energy Inefficiency':         'Relative energy inefficiency',
    'Time Weighted deltaV (mV)':            'Time weighted deltaV (mV)',
    'Capacity Weighted deltaV (mV)':        'Capacity weighted deltaV (mV)',
    'Relative Time Weighted deltaV':        'Relative time weighted deltaV',
    'Relative Capacity Weighted deltaV':    'Relative capacity weighted deltaV',
    'Ambient Temperature (°C)':             'Ambient temperature (°C)',
    'Max Charge Temperature (°C)':          'Max charge temperature (°C)',
    'Max Discharge Temperature (°C)':       'Max discharge temperature (°C)',
}

# --- DCIR metrics: internal name → Excel header (inserted before thickness) ---
dcir_column_map = {
    '10s DCR 50%SOC (mΩ)':            '10s DCR 50%SOC (mOhm)',
    'SOC Weighted 10s DCR (mΩ)':      'SOC weighted 10s DCR (mOhm)',
    'Relative 10s DCR 50%SOC':        'Relative 10s DCR 50%SOC',
    'Relative SOC Weighted 10s DCR':  'Relative SOC weighted 10s DCR',
}

# --- Thickness always last ---
thickness_column_map = {
    'Thickness (mm)':      'Thickness (mm)',
    'Relative Thickness':  'Relative thickness',
}

# --- Merge discharge DCIR summary and compute relative DCIR ---
dcir_available = dcir_enabled and not dcir_summary_all.empty
if dcir_available:
    # Drop any DCIR columns left over from a previous run of this cell.
    export_df = export_df.drop(
        columns=[c for c in dcir_column_map if c in export_df.columns]
    )

    dcir_dis = dcir_summary_all.loc[
        dcir_summary_all['Charge/Discharge'] == 'Discharge',
        ['Cell Label', 'Month', '10s DCR 50%SOC (mΩ)', 'SOC Weighted 10s DCR (mΩ)'],
    ].drop_duplicates(subset=['Cell Label', 'Month'])

    export_df = export_df.merge(dcir_dis, on=['Cell Label', 'Month'], how='left')

    # Relative DCIR = value / same cell's ref_month value.
    for _base_col, _rel_col in [
        ('10s DCR 50%SOC (mΩ)',       'Relative 10s DCR 50%SOC'),
        ('SOC Weighted 10s DCR (mΩ)', 'Relative SOC Weighted 10s DCR'),
    ]:
        _ref_vals = (
            export_df.loc[export_df['Month'] == ref_month, ['Cell Label', _base_col]]
            .drop_duplicates(subset=['Cell Label'])
            .set_index('Cell Label')[_base_col]
        )
        _ref = pd.to_numeric(export_df['Cell Label'].map(_ref_vals), errors='coerce')
        _val = pd.to_numeric(export_df[_base_col], errors='coerce')
        export_df[_rel_col] = _val / _ref.where(_ref != 0)
else:
    dcir_column_map = {}

# --- Final ordered mapping ---
final_map = {}
final_map.update(column_map)
final_map.update(dcir_column_map)
final_map.update(thickness_column_map)

# --- Report any expected columns that do not exist ---
missing_cols = [c for c in final_map if c not in export_df.columns]
if missing_cols:
    print("\n⚠️ These columns were not found in the results and will be skipped:")
    for c in missing_cols:
        print(f"     - '{c}'")
    print("   Check the names on the LEFT side of the column maps above.")
else:
    print(f"  ✅ All {len(final_map)} expected columns found.")

export_cols = [c for c in final_map if c in export_df.columns]

print(f"\n💾 Writing to '{output_filename}' ...")

with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:

    # ── One sheet per cell ────────────────────────────────────────
    for label, grp in export_df.groupby('Cell Label', sort=False):
        sheet_df = (
            grp[export_cols]
            .rename(columns=final_map)
            .fillna('NAN')
            .reset_index(drop=True)
        )
        sheet_name = str(label)[:31]
        sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)
        print(f"  ✅ {sheet_name}  ({sheet_df.shape[1]} columns, {sheet_df.shape[0]} rows)")

    # ── SOH Summary — wide-format table (cells × months) ─────────
    soh_wide = (
        export_df[['Cell Label', 'Month', 'Relative Discharge Capacity']]
        .drop_duplicates(subset=['Cell Label', 'Month'])
        .pivot(index='Cell Label', columns='Month', values='Relative Discharge Capacity')
    )
    month_order = sorted(
        soh_wide.columns,
        key=lambda x: (month_to_numeric(x) if month_to_numeric(x) is not None
                       else float('inf'))
    )
    soh_wide = soh_wide[month_order]
    soh_wide.index.name = 'Cell label'
    soh_wide.columns.name = None
    soh_wide.fillna('NAN').to_excel(writer, sheet_name='SOH Summary')
    print("  ✅ SOH Summary")

    # ── DCIR Raw sheet ────────────────────────────────────────────
    if dcir_available and not dcir_raw_all.empty:
        # Internal name → Excel header
        dcir_raw_map = {
            'test_name':        'Test name',
            'Cell Label':       'Cell label',
            'cycle':            'Cycle',
            'Charge/Discharge': 'Charge/discharge',
            'SOC_label':        'SOC label',
            'SOC':              'SOC',
            'V_0s':             'V_0s (V)',
            'V_1s':             'V_1s (V)',
            'V_10s':            'V_10s (V)',
            'V_60s':            'V_60s (V)',
            'ΔV_1s':            'ΔV_1s (V)',
            'ΔV_10s':           'ΔV_10s (V)',
            'ΔV_60s':           'ΔV_60s (V)',
            'R_1s (mΩ)':        'R_1s (mOhm)',
            'R_10s (mΩ)':       'R_10s (mOhm)',
            'R_60s (mΩ)':       'R_60s (mOhm)',
        }
        dcir_raw_all[
            [c for c in dcir_raw_map if c in dcir_raw_all.columns]
        ].rename(columns=dcir_raw_map).fillna('NAN').to_excel(
            writer, sheet_name='DCIR Raw', index=False)
        print("  ✅ DCIR Raw")

print(f"\n✅ Export complete: '{output_filename}'")
print(f"   Sheets: one per cell + 'SOH Summary'"
      + (" + 'DCIR Raw'" if dcir_available and not dcir_raw_all.empty else ""))


💾 Preparing export to '20260925_CS2E_calendar_aging_results_100SOC_to1M_v3.xlsx' ...
  ✅ All 32 expected columns found.

💾 Writing to '20260925_CS2E_calendar_aging_results_100SOC_to1M_v3.xlsx' ...
  ✅ CV_0015_CS2E0060_  (32 columns, 2 rows)
  ✅ CV_0015_CS2E0023_  (32 columns, 2 rows)
  ✅ CV_0015_CS2E0066_  (32 columns, 2 rows)
  ✅ CV_0015_CS2E0067_  (32 columns, 2 rows)
  ✅ CV_0015_CS2E0038_  (32 columns, 2 rows)
  ✅ CV_0015_CS2E0042_  (32 columns, 2 rows)
  ✅ SOH Summary
  ✅ DCIR Raw

✅ Export complete: '20260925_CS2E_calendar_aging_results_100SOC_to1M_v3.xlsx'
   Sheets: one per cell + 'SOH Summary' + 'DCIR Raw'


In [14]:
# ================================================================
# CELL 7 — EXPORT ALL RESULTS TO EXCEL
# Output structure:
#   • One sheet per cell — one row per month, missing values as "NAN".
#   • "SOH Summary" sheet — wide-format table (cells × months).
#   • "DCIR Raw" sheet — only if dcir_enabled and data available.
# ================================================================
 
print(f"\n💾 Preparing export to '{output_filename}' ...")
 
# --- Column order for per-cell sheets (matches CaLT_Columns.xlsx) ---
output_column_order = [
    'Month',
    'Time (days)',
    'Cycle',
    'Retained discharge capacity (Ah)',
    'Recovered charge capacity (Ah)',
    'Recovered discharge capacity (Ah)',
    'Relative retained discharge capacity',
    'Relative discharge capacity',
    'Self discharge (Ah)',
    'Coulombic efficiency',
    'Retained discharge energy (Wh)',
    'Recovered charge energy (Wh)',
    'Recovered discharge energy (Wh)',
    'Relative retained discharge energy',
    'Relative eischarge energy',
    'Energy efficiency',
    'Energy inefficiency',
    'Relative energy efficiency',
    'Relative energy inefficiency',
    'Time weighted deltaV (mV)',
    'Capacity weighted deltaV (mV)',
    'Relative time weighted deltaV',
    'Relative capacity weighted deltaV',
    'Ambient temperature (°C)',
    'Max charge temperature (°C)',
    'Max discharge temperature (°C)',
    'Thickness (mm)',
    'Relative thickness',
]
 
# Merge discharge DCIR summary into main table if available.
if dcir_enabled and not dcir_summary_all.empty:
    dcir_dis = (
        dcir_summary_all[dcir_summary_all['Charge/Discharge'] == 'Discharge']
        .copy()
        .rename(columns={
            '10s DCR 50%SOC (mΩ)':       '10s DCR 50%SOC (mOhm)',
            'SOC Weighted 10s DCR (mΩ)': 'SOC weighted 10s DCR (mOhm)',
        })
    )[['Cell Label', 'Month',
       '10s DCR 50%SOC (mOhm)', 'SOC weighted 10s DCR (mOhm)']]
 
    cal_aging_summary = cal_aging_summary.merge(
        dcir_dis, on=['Cell Label', 'Month'], how='left'
    )
 
    # Compute relative DCIR here, where Cell Label and Month are guaranteed
    # to be correct. For each cell, divide by that cell's ref_month value.
    for _base_col, _rel_col in [
        ('10s DCR 50%SOC (mOhm)',       'Relative 10s DCR 50%SOC (mOhm)'),
        ('SOC weighted 10s DCR (mOhm)', 'Relative SOC weighted 10s DCR (mOhm)'),
    ]:
        _ref_map = (
            cal_aging_summary[cal_aging_summary['Month'] == ref_month]
            .set_index('Cell Label')[_base_col]
            .to_dict()
        )
        def _compute_rel(row, col=_base_col, ref=_ref_map):
            rv = ref.get(row['Cell Label'])
            v  = row[col]
            if (rv is not None and pd.notna(rv) and rv != 0 and pd.notna(v)):
                return v / rv
            return np.nan
        cal_aging_summary[_rel_col] = cal_aging_summary.apply(_compute_rel, axis=1)
 
    # Insert DCIR columns before Thickness so thickness is always last.
    _tk_idx = output_column_order.index('Thickness (mm)')
    output_column_order[_tk_idx:_tk_idx] = [
        '10s DCR 50%SOC (mOhm)',
        'SOC weighted 10s DCR (mOhm)',
        'Relative 10s DCR 50%SOC (mOhm)',
        'Relative SOC weighted 10s DCR (mOhm)',
    ]
 
# Fill NaN → "NAN" string for all output columns.
for col in output_column_order:
    if col in cal_aging_summary.columns:
        cal_aging_summary[col] = cal_aging_summary[col].fillna('NAN')
 
print(f"\n💾 Writing to '{output_filename}' ...")
 
with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
 
    # ── One sheet per cell ────────────────────────────────────────
    for label, grp in cal_aging_summary.groupby('Cell Label', sort=False):
        sheet_df = (
            grp[[c for c in output_column_order if c in grp.columns]]
            .reset_index(drop=True)
        )
        sheet_name = str(label)[:31]
        sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)
        print(f"  ✅ {sheet_name}")
 
    # ── SOH Summary — wide-format table (cells × months) ─────────
    soh_wide = (
        cal_aging_summary
        .replace('NAN', np.nan)
        [['Cell Label', 'Month', 'Relative Discharge Capacity']]
        .pivot(index='Cell Label', columns='Month', values='Relative Discharge Capacity')
    )
    # Sort columns by numeric month order.
    month_order = sorted(
        soh_wide.columns,
        key=lambda x: (month_to_numeric(x) if month_to_numeric(x) is not None
                       else float('inf'))
    )
    soh_wide = soh_wide[month_order]
    soh_wide.fillna('NAN').to_excel(writer, sheet_name='SOH Summary')
    print("  ✅ SOH Summary")
 
    # ── DCIR Raw sheet ────────────────────────────────────────────
    if dcir_enabled and not dcir_raw_all.empty:
        dcir_raw_cols = [
            'test_name', 'Cell Label', 'cycle', 'Charge/Discharge',
            'SOC_label', 'SOC',
            'V_0s', 'V_1s', 'V_10s', 'V_60s',
            'ΔV_1s', 'ΔV_10s', 'ΔV_60s',
            'R_1s (mΩ)', 'R_10s (mΩ)', 'R_60s (mΩ)',
        ]
        dcir_raw_all[
            [c for c in dcir_raw_cols if c in dcir_raw_all.columns]
        ].fillna('NAN').to_excel(writer, sheet_name='DCIR Raw', index=False)
        print("  ✅ DCIR Raw")
 
print(f"\n✅ Export complete: '{output_filename}'")
print(f"   Sheets: one per cell + 'SOH Summary'"
      + (" + 'DCIR Raw'" if dcir_enabled and not dcir_raw_all.empty else ""))


💾 Preparing export to '20260925_CS2E_calendar_aging_results_100SOC_to1M_v2.xlsx' ...

💾 Writing to '20260925_CS2E_calendar_aging_results_100SOC_to1M_v2.xlsx' ...
  ✅ CV_0015_CS2E0060_
  ✅ CV_0015_CS2E0023_
  ✅ CV_0015_CS2E0066_
  ✅ CV_0015_CS2E0067_
  ✅ CV_0015_CS2E0038_
  ✅ CV_0015_CS2E0042_
  ✅ SOH Summary
  ✅ DCIR Raw

✅ Export complete: '20260925_CS2E_calendar_aging_results_100SOC_to1M_v2.xlsx'
   Sheets: one per cell + 'SOH Summary' + 'DCIR Raw'
